In [9]:
# !conda init powershell 
# !conda activate tree-mes

In [10]:
# !pip install statsmodels

In [11]:
# !pip install xgboost

In [12]:
# !pip install optuna

In [13]:
# !pip install seaborn --upgrade

In [14]:
# !pip install -U scikit-learn
# !pip install wandb
# !pip install shap
# !pip install tifffile
# !pip install imagecodecs
# !pip install numba.core
# import shap
# !pip install opencv-python
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [15]:
import pickle
import wandb
import networkx as nx

In [16]:
from os import path,listdir,mkdir,rename,chdir
import time
from datetime import datetime
import json
from math import floor
import re

In [17]:
import pandas as pd
pd.options.mode.chained_assignment = None
pd.set_option('max_rows', 100)
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split,GroupShuffleSplit,GroupKFold
from sklearn.preprocessing import (MinMaxScaler,StandardScaler,PowerTransformer,
                                   PolynomialFeatures,OneHotEncoder,RobustScaler,QuantileTransformer)
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV,ElasticNetCV, SGDRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn import model_selection
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error,make_scorer,mean_squared_log_error
import time
from sklearn.model_selection import KFold, cross_validate, StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
import optuna

from sklearn.decomposition import PCA

In [18]:
# from tensorflow.keras.layers import Input,LeakyReLU
# from tensorflow.keras import Model,callbacks, optimizers,regularizers
# from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Activation
# from tensorflow.keras.optimizers import Adam,RMSprop,Adadelta,Adagrad,Adamax,Nadam,Ftrl,SGD
# from tensorflow.keras.backend import clear_session
# from tensorflow.keras.models import clone_model

In [19]:
from feature_extractor import *

ModuleNotFoundError: No module named 'vision'

In [ ]:
f_csv_path = "/media/fruitspec-lab/easystore/JAIZED_CaraCara_151122/Summer gold experiment tree count.csv"
f_df = pd.read_csv(f_csv_path)

In [ ]:
f_df.columns

In [ ]:
f_df = pd.melt(f_df,id_vars = ["Row/tree"],value_vars = [str(i+1) for i in range(12)],value_name = "F",var_name = "Row")

In [ ]:
row2_fixing_bool = (f_df["Row"] == '2') & (f_df["Row/tree"] > 27)

In [ ]:
new_2_vals =  f_df[row2_fixing_bool]["Row/tree"].apply(lambda x: x+1).values
f_df.loc[row2_fixing_bool, "Row/tree"] += 1

In [ ]:
f_df["tree_name"] = f_df.apply(lambda x: f'R{x["Row"]}_T{x["Row/tree"]}',axis = 1)

In [ ]:
f_df_clean = f_df[f_df["F"].notna()].reset_index()[["tree_name","F"]]

In [ ]:
f_df_clean.to_csv("/media/fruitspec-lab/easystore/JAIZED_CaraCara_301122/df_f.csv")

In [ ]:
scalers = {"Standard":StandardScaler(),
            "MinMax":MinMaxScaler(),
            "Power":PowerTransformer(),
          "Quantile":QuantileTransformer(),
          "Robust":RobustScaler()}

In [ ]:
np.random.seed(2)
tree_images =  {"1": {"fsi":np.random.normal(size = (50,50,3)), "rgb":np.random.normal(size = (50,50,3)),"zed":np.random.normal(size = (50,50,3)),
                     "swir_975":np.random.normal(size = (50,50,3)),"nir":np.random.normal(size = (50,50,3))},
                            "2": {"fsi":np.random.normal(size = (50,50,3)), "rgb":np.random.normal(size = (50,50,3)),"zed":np.random.normal(size = (50,50,3)),
                                 "swir_975":np.random.normal(size = (50,50,3)),"nir":np.random.normal(size = (50,50,3))}} 
tracker_results ={"1": {str(i):(tuple(np.random.random_integers(0,10,size = 2)),
                                tuple(np.random.random_integers(0,10,size = 2))) for i in range(50)},
                  "2": {str(i):(tuple(np.random.random_integers(0,10,size = 2)),
                                tuple(np.random.random_integers(0,10,size = 2))) for i in range(50)}}
tree_name = ""

In [ ]:
phisical_features_names = init_physical_parmas([]).keys()
location_features_names = ['mst_sums_arr','mst_mean_arr','mst_skew_arr','clusters_area_mean_arr',
                         'clusters_area_med_arr','clusters_ch_area_mean_arr','clusters_ch_area_med_arr',
                         'n_clust_arr_2','n_clust_arr_4','n_clust_arr_6','n_clust_mean_arr','n_clust_med_arr']
tree_features_names = init_fruit_params([]).keys()
v_i_features = transform_to_vi_features({**get_additional_vegetation_indexes(0, 0, 0, fill=[0])}).keys()
all_cols = list(phisical_features_names) + list(location_features_names) + list(tree_features_names) + list(v_i_features)
drop_cols_vi = ["tdvi","gci","msavi2","tgi","gndvi", "gosavi", "dvi","gndvi","gdvi","gemi","lai","msr","ndre","vari",
                "grvi", "rdvi", "savi", "arvi", "ipvi", "sr","mnli", "wdrvi", "evi", "gari", "pvi"]
drop_cols = ["q1","q3","total_time","n_clust_arr_4","n_clust_arr_6","n_clust_mean_arr","n_clust_med_arr","frame"]
final_cols = list(set(all_cols) - set(drop_cols) - set(drop_cols_vi) - {f"col_skew" for col in drop_cols_vi})
drop_final = list(set(drop_cols_vi + [f"{col}_skew" for col in drop_cols_vi] + drop_cols))
drop_sides = ["B"]
drop_trees = ["R4_T2", "R4_T4"]



In [ ]:
full_names_to_drop = []
hidden_range = (-0.1,2)
cv_range = (5,np.inf)
F_range = (0,2500)
fruits_exclude = ["apple"]

In [ ]:
csv_path = "/media/fruitspec-lab/easystore/JAIZED_CaraCara_301122/plot_features_5_pysic_features.csv"
# csv_path = [f"/media/fruitspec-lab/easystore/JAIZED_CaraCara_301122/R{i}/trees/row_features.csv" for i in range(2,8)]
# csv_path = "/media/fruitspec-lab/easystore/JAIZED_CaraCara_301122/plot_features12_12.csv"
# TODO add F

# clean unwanted trees    
clean_set = pd.DataFrame()
if isinstance(csv_path, type(list())):
    for row_csv_path in csv_path:
        clean_set = pd.concat([clean_set,pd.read_csv(row_csv_path)],ignore_index = True)
    clean_set.reset_index(drop = True, inplace = True)
else:
    clean_set = pd.read_csv(csv_path)
df_cols = clean_set.columns
if "% hidden" in df_cols:
    clean_set = clean_set[clean_set["% hidden"] > hidden_range[0]]
    clean_set = clean_set[clean_set["% hidden"] < hidden_range[1]]
clean_set = clean_set[clean_set["cv"] > cv_range[0]]
clean_set = clean_set[clean_set["cv"] < cv_range[1]]
if "F" in df_cols:
    clean_set = clean_set[clean_set["F"] > F_range[0]]
    clean_set = clean_set[clean_set["F"] < F_range[1]]
if "fruit_type" not in df_cols:
    clean_set["fruit_type"] = "orange"
clean_set = clean_set[clean_set["fruit_type"].map(lambda x: x not in fruits_exclude)]
if "full_name" in df_cols:
    clean_set = clean_set[clean_set["full_name"].map(lambda x: x not in full_names_to_drop)]
if "Unnamed: 0" in df_cols:
    clean_set.drop("Unnamed: 0",axis = 1,inplace = True)
ohe = OneHotEncoder()
ohe_res = ohe.fit_transform(clean_set["fruit_type"].values.reshape(-1, 1)).toarray()
clean_set[ohe.categories_[0].tolist()[0]] = ohe_res

In [ ]:
if "block_name" not in clean_set.columns:
    clean_set["block_name"] = "CaraCaraNir"
if "side" not in clean_set.columns:
    clean_set["side"] = "A"

In [ ]:
clean_set["tree_number"] = clean_set["name"].apply(lambda x: int(x.split("_")[-1][1:]))
nircaracara_bool = clean_set["block_name"] == "CaraCaraNir"
B_side_bool = clean_set["tree_number"] > 48
clean_set["side"][(nircaracara_bool) & B_side_bool] = "B"
clean_set["name"][(nircaracara_bool) & B_side_bool] = clean_set["name"][(nircaracara_bool) & B_side_bool].apply(lambda x: f"{x.split('_')[0]}_T{100 - int(x.split('_')[-1][1:])+1}")
clean_set.drop([col for col in clean_set.columns if col.startswith("Unnamed")]+drop_final, axis = 1, inplace = True)

In [ ]:
clean_set

In [ ]:
clean_set["F"] = clean_set["name"].map(dict(zip(f_df["tree_name"],f_df["F"])))
clean_set = clean_set[np.isfinite(clean_set["F"])]
clean_set.replace("[0]",0, inplace = True)
change_cols = ['gli','gli_skew','ndri','ndri_skew','ndvi','ndvi_skew','nli','nli_skew','sipi','sipi_skew']
clean_set[change_cols] = clean_set[change_cols].astype(float)
clean_set = clean_set[clean_set["side"].apply(lambda x: x not in drop_sides)]
clean_set = clean_set[clean_set["name"].apply(lambda x: x not in drop_trees)]
clean_set[["name","side","F","cv"]].sort_values("F")

In [ ]:
clean_set["cv/F"] = clean_set["cv"]/clean_set["F"]
clean_set[["name","side","F","cv", "height","width","cv/F"]].sort_values("cv/F")[:20]

In [ ]:
clean_set["height/F"] = clean_set["height"]/clean_set["F"]
clean_set[["name","side","F","cv", "height","width","avg_volume","height/F"]].sort_values("height/F")[-20:]

In [ ]:
clean_set.drop(["height/F", "cv/F"],axis =1, inplace = True)

In [ ]:
numer_cols = list(clean_set.select_dtypes(include=np.number).columns)
cat_cols = [col for col in clean_set.select_dtypes(exclude=np.number).columns if col != "full name"] 

In [ ]:
clean_set[clean_set["clusters_area_med_arr"] >4007][["name","side","F","cv", "height","width","avg_perimeter"]]

In [ ]:
# clean_set.drop([18,22], inplace = True)

### plots

In [ ]:
coludy_rows = ["R8", "R9"]
clean_set["cloudy"] = clean_set["name"].apply(lambda x:  np.any([row in x for row in coludy_rows]) )
sns.set(font_scale=1)
for col in numer_cols:
    sns.lmplot(data = clean_set,x = col,y="F",hue= "cloudy", scatter_kws={'s':2}, order = 1)
    plt.show()

In [ ]:
clean_set["row"] = clean_set["name"].apply(lambda x: x.split("_")[0])
sns.set(font_scale=1)
for col in numer_cols:
    ax = sns.lmplot(data = clean_set,x = col,y="F",hue= "row", scatter_kws={'s':2}, order = 1)
    sns.regplot(data = clean_set,x = col,y="F", scatter_kws={'s':2}, order = 1,ci=0, ax=ax.axes[0,0],
               x_ci = 0, color = "black", line_kws={"ls":"--"},scatter=False)
    plt.show()
clean_set.drop(["row"],axis =1, inplace = True)

In [ ]:
sns.set(font_scale=1)
for col in numer_cols:
    sns.lmplot(data = clean_set,x = col,y="F",hue= "fruit_type", scatter_kws={'s':2})
    plt.show()
clean_set.drop("fruit_type",axis = 1,inplace = True)

In [ ]:
col = "clusters_area_med_arr"
clean_df_temp = clean_set.copy(True)
clean_df_temp = clean_df_temp[clean_df_temp[col]<4000]
clean_df_temp[f"{col}_cv"] = clean_df_temp[col]*clean_df_temp["cv"]
sns.lmplot(data = clean_df_temp,x = col,y="F", scatter_kws={'s':2})
plt.show()
sns.lmplot(data = clean_df_temp,x = f"{col}_cv",y="F", scatter_kws={'s':2})
plt.show()

In [ ]:
for_corr_plot = set([col for col in numer_cols if not col.endswith("skew")]) - {"apple",
                                                                                "mandarin","lemon","orange"}
corr_frame = clean_set[for_corr_plot].corr()
plt.figure(figsize=(30,15))
sns.heatmap(corr_frame, 
        xticklabels=corr_frame.columns,
        yticklabels=corr_frame.columns,annot = True, annot_kws={"fontsize": 12})


In [ ]:
sns.set(font_scale=1.75)
n_cols = corr_frame.shape[0]
half_cols = int(n_cols/2)
plt.figure(figsize=(30,15))
sns.heatmap(corr_frame.iloc[:half_cols,:half_cols], 
        xticklabels=corr_frame.columns[:half_cols],
        yticklabels=corr_frame.columns[:half_cols],annot = True, annot_kws={"fontsize": 12})
plt.show()
plt.figure(figsize=(30,15))
sns.heatmap(corr_frame.iloc[:half_cols,half_cols:], 
        xticklabels=corr_frame.columns[half_cols:],
        yticklabels=corr_frame.columns[:half_cols],annot = True, annot_kws={"fontsize": 12})
plt.show()
plt.figure(figsize=(30,15))
sns.heatmap(corr_frame.iloc[half_cols:,:half_cols], 
        xticklabels=corr_frame.columns[:half_cols],
        yticklabels=corr_frame.columns[half_cols:],annot = True, annot_kws={"fontsize": 12})
plt.show()
plt.figure(figsize=(30,15))
sns.heatmap(corr_frame.iloc[half_cols:,half_cols:], 
        xticklabels=corr_frame.columns[half_cols:],
        yticklabels=corr_frame.columns[half_cols:],annot = True, annot_kws={"fontsize": 12})
plt.show()

In [ ]:
corr_frame_stacked = corr_frame.stack().to_frame()
bool_vec = np.all([np.abs(corr_frame_stacked[0].values) > 0.8 , np.abs(corr_frame_stacked[0].values) < 1],axis = 0)
for i, row in corr_frame_stacked[bool_vec].iterrows():
    var1, var2 = row.name
    sns.lmplot(data = clean_set,x = var1,y=var2, scatter_kws={'s':2})
    plt.show()

### PCA

In [ ]:
rows = clean_set["name"].apply(lambda x: x.split("_")[0])
X_tr_lr = clean_set.copy()
y = X_tr_lr['F']
X_tr_lr.drop("F",axis = 1,inplace = True)
X_tr_lr.drop(["block_name","side","tree_number", "name"],axis = 1,inplace = True)
X_tr_lr['cv^2'] = X_tr_lr["cv"]**2
X_tr_lr['1/mst_mean_arr'] = 1/(X_tr_lr["mst_mean_arr"]+1)
X_tr_lr['mst_sums_arr^2'] = X_tr_lr["mst_sums_arr"]**2
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold()
X_tr_lr_selected = selector.fit_transform(X_tr_lr)
X_tr_lr = pd.DataFrame(X_tr_lr_selected,columns = X_tr_lr.columns[selector.get_support()])

In [ ]:
rows.reset_index(inplace = True, drop = True)
X_tr_lr.reset_index(inplace = True, drop = True)
y.reset_index(inplace = True, drop = True)
X_tr_lr.fillna(0, inplace = True)

In [ ]:
n_comp = None
pca = PCA(n_comp)
X_tr_lr_sc = StandardScaler().fit_transform(X_tr_lr)
x_pcaed = pca.fit_transform(X_tr_lr_sc, y)
x_pcaed = pd.DataFrame(x_pcaed,columns = [f"PC_{i+1}" for i in range(n_comp if n_comp != None else X_tr_lr.shape[1])])

In [ ]:
max_comp = 20
plt.figure(figsize=(15,15))
plt.plot(range(1,max_comp+1),pca.explained_variance_ratio_[:max_comp])
plt.plot(range(1,max_comp+1),np.cumsum(pca.explained_variance_ratio_[:max_comp]))
plt.axhline(0.9)
plt.show()
pd.DataFrame(np.abs(pca.components_.T), index = X_tr_lr.columns).sort_values(2,ascending = False)

In [ ]:
# X_tr_lr[x_pcaed.columns] = x_pcaed

### modeling

In [ ]:
def mape(y_true,y_pred,trees):
    df = pd.DataFrame({"F":y_true,"prediction_F":y_pred,"trees":trees})
    agg_table = pd.pivot_table(df,index = "trees",values = ["prediction_F","F"],aggfunc = np.mean)
    return np.mean(np.abs(agg_table["F"]- agg_table["prediction_F"]) / agg_table["F"])

def mape_scoring(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred)/y_true)

def naive_prediction(data,frames = True):
    df = data.copy(True)
    if frames:
        df = df[df["frame"] == 1] 
    fruit_avg = {col: df["F"][df[col] == 1].mean()
                 for col in ["orange","lemon","mandarin","apple"] if col in df.columns}
    print(fruit_avg)
    predictions = data[fruit_avg.keys()] * fruit_avg
    return predictions.max(axis =1)

def naive_score(data,frames = True):
    predictions = naive_prediction(data,frames)
    return mape_scoring(data["F"],predictions)
print(naive_score(clean_set,frames = False))

In [ ]:
class model_hybrid_selector(BaseEstimator):
    
    def __init__(self, base_estimator=LinearRegression(), params = {},
                 scoring="neg_mean_absolute_percentage_error", statisticly = False):
        print(params)
        if len(params)>0:
            self.model = base_estimator(**params)
        else:
            self.model = base_estimator
        self.cols = []
        self.scoring= scoring
        self.stratify = None
        self.statisticly = statisticly

    def get_score(self,X_tr,y_tr,cols,cv=16, n_jobs=-1,random_state = None):
        if not isinstance(self.stratify, type(None)):
            cv = StratifiedKFold(cv, shuffle= True,random_state=random_state)
        score = cross_validate(self.model, X_tr[cols] , y=y_tr,cv=cv, n_jobs=n_jobs, 
                                                        scoring=self.scoring)
        test_scores = score["test_score"]
        return np.mean(test_scores * (-1)), np.std(test_scores)

    def get_candidate(self,results_dict, results_stds, cur_score):
        sort_by = "z_score" if self.statisticly else "new_result"
        new_results = list(results_dict.values())
        z_score = (np.array(new_results) - cur_score) / np.array(results_stds)
        df = pd.DataFrame({"feature": list(results_dict.keys()),
                           "new_result": list(results_dict.values()),
                           "z_score": z_score}).sort_values(sort_by)
        return df.iloc[0] if self.statisticly else df.iloc[0]

    def print_results(self,canidate,cur_score,cols,added = True):
        change = 0
        status = "added" if added else "removed"
        new_score = canidate["new_result"]
        if np.isnan(new_score):
            return cur_score,change,cols
        if new_score < cur_score:
            feature = canidate["feature"]
            if added:
                cols.append(feature)
            elif feature in cols:
                cols.remove(feature)
            change = 1
            print(f"{status} {feature}, new_score: {new_score}")
        else:
            print(f"no feature could be {status}, new_score: {new_score}")
        return new_score,change,cols   
    
    def step(self,X_tr,y_tr,use_alwyas,cols,cur_score, forward = True):
        results_dict, results_stds, change = {}, [], 0
        canidate_cols = set(X_tr.columns) - set(use_alwyas) - set(cols) if forward else set(cols) - set(use_alwyas)
        if len(canidate_cols) == 0:
            return cols,change,cur_score 
        random_state = np.random.randint(0,1000,1)[0]
        assured_cols = cols + use_alwyas
        for col in tqdm(canidate_cols):
            train_cols =  assured_cols + [col] if forward else set(assured_cols) - {col}
            results_dict[col], cur_std = self.get_score(X_tr, y_tr, train_cols, random_state=random_state)
            results_stds.append(cur_std)
        candidate = self.get_candidate(results_dict, results_stds, cur_score)
        new_score,change,cols = self.print_results(candidate,cur_score,cols,added = forward)
        return cols,change, min(new_score,cur_score)

    def hybrid_stepwise_selection(self,X_tr,y_tr,num_stuck = 3,cols = [], use_alwyas = []):
        no_change, cur_score = 0, np.inf
        print(cur_score)
        while no_change < num_stuck:
            cols,added_result,cur_score = self.step(X_tr,y_tr,use_alwyas,cols,cur_score)
            cols,removed_result,cur_score = self.step(X_tr,y_tr,use_alwyas,cols,cur_score, forward = False)
            if added_result or removed_result:
                no_change = 0
            else:
                no_change += 1
            print(f"score: {cur_score},cols: {cols}")
        return cols + use_alwyas,cur_score
    
    def fit(self,X_tr,y_tr,num_stuck = 3,cols = [], use_alwyas = [], stratify = None):
        self.stratify = stratify
        self.cols,cur_score = self.hybrid_stepwise_selection(X_tr,y_tr,num_stuck,cols,use_alwyas)
        self.model.fit(X_tr[self.cols],y_tr)
        return cur_score

    def fitplusplus(self,X_tr,y_tr,num_stuck = 3, stratify = None,n_iter = 15):
        self.stratify = stratify
        all_cols_list = list(X_tr.columns)
        n_cols = len(all_cols_list)
        cols_arr = {}
        for i in range(1,n_iter+1):
            if i == 1:
                cols = []
            elif i == 2:
                cols = all_cols_list.copy()
            else:
                size = np.random.randint(1,n_cols-1)
                cols = np.random.choice(all_cols_list.copy(),size = size,replace = False).tolist()
            self.fit(X_tr,y_tr,num_stuck = num_stuck,cols = cols,stratify = stratify)
            cols_arr[i] = {"cols": self.cols, "score": self.get_score(X_tr,y_tr,self.cols, n_jobs=-1)}
            print(self.cols)
            print(i)
        cols_arr_sorted = {k: v for k, v in sorted(cols_arr.items(), key=lambda item: item[1]["score"])}
        best_iteration = cols_arr[list(cols_arr_sorted.keys())[0]]
        self.cols = best_iteration["cols"]
        return best_iteration["score"]
        
    def predict(self,X_pred):
        return self.model.predict(X_pred[self.cols])
    
    def fit_predict(self,X_tr,y_tr,num_stuck = 3,cols = [], use_alwyas = [], stratify = None):
        self.fit(X_tr,y_tr,num_stuck,cols,use_alwyas,stratify)
        return self.predict(X_tr)
    
    def iterative_fitting(self,X_tr,y_tr,num_stuck = 3,cols = [], use_alwyas = [], stratify = None,n_iter = 15,
                         sampling = 0.8):
        self.stratify = stratify
        cols_arr = {}
        for i in range(1,n_iter+1):
            self.fit(X_tr,y_tr,num_stuck = num_stuck,cols = cols,stratify = stratify)
            cols = self.cols
            if i > 1:
                cols_arr_sorted = {k: v for k, v in sorted(cols_arr.items(), key=lambda item: item[1]["score"])}
                best_iteration = cols_arr[list(cols_arr_sorted.keys())[0]]
                cols = best_iteration["cols"]
            cols = np.random.choice(np.array(cols),replace = False,size=int(sampling*len(cols))).tolist()
            cols_arr[i] = {"cols": self.cols, "score": self.get_score(X_tr,y_tr,self.cols, n_jobs=-1)}
            print(self.cols)
            print(i)
        cols_arr_sorted = {k: v for k, v in sorted(cols_arr.items(), key=lambda item: item[1]["score"])}
        best_iteration = cols_arr[list(cols_arr_sorted.keys())[0]]
        self.cols = best_iteration["cols"]
        return best_iteration["score"]


### baseline

In [ ]:
# TODO add cols
sc = scalers["Power"]

def cross_validate_with_mean(model = None, X = None,y = None,cv = 16, groups = None, ret_preds = False):
    results = []
    tree_res = []
    X = X.to_numpy()
    all_preds = np.zeros(len(y))
    if not isinstance(y, type(np.array([]))):
        y = y.to_numpy()
    if not isinstance(groups,type(None)):
        cv = groups.nunique()
        kf = GroupKFold(n_splits=cv)
        iterable_kf = kf.split(X,y,groups)
    else:
        kf = KFold(n_splits=cv, random_state=43, shuffle=True)
        iterable_kf = kf.split(X)
    for train_index, test_index in iterable_kf:
        x_train, x_test, y_train, y_test = X[train_index] , X[test_index], y[train_index] , y[test_index]
        #x_train, x_test, y_train, y_test = X[test_index] , X[train_index], y[test_index] , y[train_index]
        if isinstance(model,type(None)):
            y_pred = np.array([np.mean(y_train)]*len(y_test))
        else:
            model.fit(x_train,y_train)
            y_pred = model.predict(x_test)
        y_true_sum = y_test.sum()
        results.append(abs(y_pred.sum() - y_true_sum)/(y_true_sum))
        tree_res.append(np.mean(abs(y_pred-y_test)/(y_test)))
        test_group = ""
        if not isinstance(groups,type(None)):
            test_group = f"({groups[test_index[0]]})"
        y_pred_sum = y_pred.sum()
        acc = np.abs(y_true_sum-y_pred_sum)/y_true_sum
        all_preds[test_index] = y_pred
        print(F"true: {y_true_sum},    pred: {y_pred_sum}. ({acc*100 :.2f} %) {test_group}" )
    print(np.mean(tree_res), np.std(tree_res))
    if ret_preds:
        return all_preds
    return np.mean(results), np.std(results)

print(cross_validate_with_mean(None,X_tr_lr,y))

print(cross_validate_with_mean(None,X_tr_lr,y,groups = rows))

### fit linear

In [ ]:
mhs = model_hybrid_selector()
mhs.iterative_fitting(X_tr_lr,y,num_stuck = 3,cols = [],use_alwyas = [])
# mhs.fit(X_tr_lr,y,num_stuck = 3,cols = [],use_alwyas = [])


In [ ]:
mhs.cols

In [ ]:
print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhs.cols],y))

print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhs.cols],y,groups = rows))

In [ ]:
mhsfpp = model_hybrid_selector()
mhsfpp.fitplusplus(X_tr_lr,y,num_stuck = 3,n_iter = 50)
print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp.cols],y))

print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp.cols],y,groups = rows))

In [ ]:
mhsfpp_statisticly = model_hybrid_selector(statisticly = True)
mhsfpp_statisticly.fitplusplus(X_tr_lr,y,num_stuck = 3)
print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp_statisticly.cols],y))

print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp_statisticly.cols],y,groups = rows))


In [ ]:
cols = mhsfpp.cols
X_non_scale = X_tr_lr.copy()[cols]
sc = scalers["Power"]
X_scaled = sc.fit_transform(X_non_scale)
X_scaled = pd.DataFrame(X_scaled,columns = X_non_scale.columns)
X_tr_lm = sm.add_constant(X_scaled)

model = sm.OLS(y.reset_index(drop=True),X_tr_lm)
results = model.fit()
results.summary()

In [ ]:
y_preds = cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp.cols],y,groups = rows, ret_preds =True)
clean_set["pred"] = y_preds
clean_set["err"] = y_preds - clean_set["F"]

In [ ]:
sns.kdeplot(data = clean_set, x = "err")

In [ ]:
mhsfpp.cols

In [ ]:
clean_set["abs_err"] = clean_set["err"].abs()
clean_set.sort_values("abs_err")[mhsfpp.cols[1:] + ["name","pred","F","err"]][-20:]

### QuantileRegressor

In [ ]:
# mhsfpp_qr = model_hybrid_selector(QuantileRegressor(solver = "highs" ,alpha = 0))
# mhsfpp_qr.fitplusplus(X_tr_lr,y,num_stuck = 3)
# print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp_qr.cols],y))

# print(cross_validate_with_mean(LinearRegression(),X_tr_lr[mhsfpp_qr.cols],y,groups = rows))

### fit ploy linear

In [ ]:
# poly = PolynomialFeatures(interaction_only=False)
# cols = mhs.cols

# X_inteact = pd.DataFrame(poly.fit_transform(X_tr_lr[cols]),
#                          columns = poly.get_feature_names(cols))
# mhs_interact =   model_hybrid_selector(LinearRegression())          
# mhs_interact.fit(X_inteact, y, num_stuck = 3, cols = mhs.cols,use_alwyas = [])

### fit scalers

In [ ]:
# for sclaer in scalers.keys():
#     print(sclaer)
#     sc = scalers[sclaer]
#     pipe = Pipeline(steps = [(sclaer, sc),
#                             ("lm",LinearRegression())])
#     print(model_hybrid_selector(pipe).fit(X_tr_lr,y,num_stuck = 7,cols = [],use_alwyas = ["cv"]))

### XGB

In [ ]:
#             'min_child_weight': trial.suggest_float('min_child_weight',1e-1, 2),
#             'max_depth': trial.suggest_int('max_depth',1,9),
#             'gamma': trial.suggest_float("gamma",1e-3, 1),
#             'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1, step=0.1),
#             'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3, 1, step=0.1),
#             "reg_alpha" : trial.suggest_float("reg_alpha" ,0.1, 5),
#             "reg_lambda" : trial.suggest_float("reg_lambda" ,0.1, 5),
#             'subsample': trial.suggest_float('subsample', 0.4, 0.9, step=0.1),

xgb_params = {'eta': 0.016811610093812577,  'min_child_weight': 1.1759101569784651,
  'max_depth': 8,
  'gamma': 0.9442254921832527,
  'colsample_bytree': 0.4,
  'colsample_bylevel': 0.8,
  'reg_alpha': 4.623105604488696,
  'reg_lambda': 3.9583199400391553,
  'subsample': 0.8,
  'n_estimators': 150}
model_xgb = XGBRegressor(**xgb_params)
score = cross_validate(model_xgb, X_tr_lr , y=y,cv=8, n_jobs=-1, scoring="neg_mean_absolute_percentage_error")

In [ ]:
print((np.mean(score['test_score']), np.std(score['test_score'])))
print(cross_validate_with_mean(model_xgb,X_tr_lr,y))
print(cross_validate_with_mean(model_xgb,X_tr_lr,y,groups = rows))

### feature importance

In [ ]:
from sklearn.inspection import permutation_importance
model_xgb = XGBRegressor(**xgb_params)
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X_tr_lr,y,test_size = 0.2)
model_xgb.fit(X_train_lr,y_train_lr)

result = permutation_importance(model_xgb, X_test_lr, y_test_lr, n_repeats=16, n_jobs=8)
sorted_idx = result.importances_mean.argsort()

In [ ]:
permutation_importance_res = pd.DataFrame({"names": X_train_lr.columns,"imp": result.importances_mean,
                                          "std": result.importances_std})
permutation_importance_res["l_ci"] = permutation_importance_res["imp"] - 2* permutation_importance_res["std"] 
permutation_importance_res= permutation_importance_res.sort_values("imp",ascending = False)

In [ ]:
permutation_importance_res

In [ ]:
imp_params = permutation_importance_res["names"][permutation_importance_res["imp"]> 0].values
model_xgb = XGBRegressor(**xgb_params)
model_xgb.fit(X_tr_lr[imp_params],y)
print((np.mean(score['test_score']), np.std(score['test_score'])))
print(cross_validate_with_mean(model_xgb,X_tr_lr[imp_params],y))
print(cross_validate_with_mean(model_xgb,X_tr_lr[imp_params],y,groups = rows))

### optuna

In [ ]:
# TODO try RFE with XGV as a combiend pipeline for paramater search
def objective_xgb(trial,n_jobs = -1, scoring = "neg_mean_absolute_percentage_error",
                 n_splits=8, shuffle=True, random_state=42, param = None, stratify = None):
    if not isinstance(stratify, type(None)):
        kfolds = StratifiedKFold(cv, shuffle= shuffle, random_state=random_state)
    else:
        kfolds = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    if isinstance(param, type(None)):
        param = {
            'eta': trial.suggest_float("eta", 0.0001,0.5),
            'min_child_weight': trial.suggest_float('min_child_weight',1e-1, 5),
            'max_depth': trial.suggest_int('max_depth',1,20),
            'gamma': trial.suggest_float("gamma",1e-3, 1),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1, step=0.1),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3, 1, step=0.1),
            "reg_alpha" : trial.suggest_float("reg_alpha" ,0.1, 5),
            "reg_lambda" : trial.suggest_float("reg_lambda" ,0.1, 5),
            'subsample': trial.suggest_float('subsample', 0.4, 0.9, step=0.1),
            'n_estimators': trial.suggest_int('n_estimators', 10, 2000, step = 10)

        }
    reg = XGBRegressor(**param)
    scores = cross_val_score(reg, X_tr_lr,y, cv=kfolds,scoring=scoring)
    return scores.mean()

In [ ]:
def optimize_estimator(objective,direction='maximize',n_trails = 200,save_int = 0, save_dst="", read_from=""):
    if read_from != "":
        study = joblib.load(read_from)
    else:
        study = optuna.create_study(direction=direction)
    if save_int == 0:
        study.optimize(objective, n_trials=n_trails, n_jobs = -1)
        if save_dst!="":
            joblib.dump(study, save_dst)
    else:
        trails_per_save = np.ceil(n_trails/save_int)
        for i in range(save_int):
            study.optimize(objective, n_trials=trails_per_save)
            if save_dst!="":
                joblib.dump(study, save_dst)
    best_params = study.best_params
    best_score = study.best_value
    print(f"Best score: {best_score}\n")
    print(f"Optimized parameters: {best_params}\n")
    return best_params, best_score

optimize_estimator(objective_xgb,n_trails=5000,save_int = 100,save_dst = "study_ndvi_fix.pkl")

In [ ]:
from sklearn.linear_model import LassoCV
las = LassoCV(n_alphas=1000,cv = 8, n_jobs = -1,max_iter=2000)
sc = StandardScaler()
las.fit(pd.DataFrame(sc.fit_transform(X_train_lr), columns = X_train_lr.columns)
        ,y_train_lr)

las_score = cross_validate(las, sc.fit_transform(X_train_lr) ,
               y=y_train_lr,cv=8, n_jobs=-1, scoring="neg_mean_absolute_percentage_error")

In [ ]:
np.mean(las_score["test_score"])

In [ ]:
plt.plot(las.alphas_,np.mean(las.mse_path_,axis = 1))